# Telegram Movie1 — Kaggle compact fix (below 1 MB)

This fixes the Kaggle **"kernel source must be less than 1 megabytes"** error. The notebook embeds a more-efficient compressed bundle with the same **3,780 prepared outgoing messages, 3,751 photo references, all non-channel download links, uploader code, original post order and checksums**. The redundant caption audit was omitted only to shrink the notebook; outgoing captions and links are unchanged.

**Destination from your previous notebook:** `Bollywood Movie` (`-1004492918137`). **Private invite you provided:** `https://t.me/+9GdbVBmx_3dhNWM9`.

Your screenshot confirms the bot can see this channel, but not the marker post. This new version tries Telegram's primary invite link and recent updates first; if neither is accessible, it uses the **previously configured exact channel ID** only if Telegram also confirms the channel title and the bot's posting-admin permission. **It cannot independently prove that the invite URL belongs to that channel in this fallback.** Make sure that *Bollywood Movie* is your intended destination; the notebook never selects some other accessible channel.

### Run once

1. Import this **new smaller notebook** into Kaggle (don't keep running the old oversized notebook). Enable Internet and CPU.
2. Grant access to your existing `TELEGRAM_BOT_TOKEN` secret (checkbox). The bot must be a posting admin of `Bollywood Movie`.
3. Choose **Save Version → Save & Run All**. Default is `SEND_ALL`. It starts posting without interactive prompts if the channel checks pass. Once a committed/background run has started, you can close the browser.

**No need to repost the verification marker.** No need to enter the channel ID. Stop any other active Telegram migrations first.

### Restart safety

If you've already sent migration posts before, **attach the matching prior `progress_-1004492918137.json` file as a Kaggle input before starting**. Otherwise it may repost previous messages. Kaggle temporary files don't guarantee recovery after a failed run. A GitHub link in Kaggle doesn't automatically back up progress or create a long-running server. For reliable restarts, use a persistent server with durable checkpoints. If uncertain, do not start another upload until the channel history and progress have been reconciled.

This notebook makes API requests only when run on Kaggle; local tests did not send Telegram posts.


In [ ]:
#@title Kaggle settings — everything is pre-filled (NO edits needed)
TARGET_INVITE_URL = 'https://t.me/+9GdbVBmx_3dhNWM9'
VERIFICATION_POST = 'MIGRATION_TARGET_9GdbVBmx_3dhNWM9'
PREVIOUS_NOTEBOOK_CHAT_ID = -1004492918137
EXPECTED_TARGET_TITLE = 'Bollywood Movie'
# The previous Colab notebook already selected this channel by ID; screenshots
# show that the same bot can access it. If the marker is unavailable, use that
# saved ID ONLY after verifying exact title and bot posting-admin privileges.
ALLOW_PREVIOUS_NOTEBOOK_DESTINATION = True
RUN_MODE = 'SEND_ALL'  #@param ['SEND_ALL','SEND_5','PREVIEW']
assert RUN_MODE in {'SEND_ALL','SEND_5','PREVIEW'}
print('RUN_MODE:',RUN_MODE,'| pinned destination:',PREVIOUS_NOTEBOOK_CHAT_ID,
      '| expected title:',EXPECTED_TARGET_TITLE)


In [ ]:
#@title 1. Prepare the exact migration data (NO Telegram sends)
from pathlib import Path
import base64, hashlib, importlib.util, io, json, lzma, sys, tarfile

WORKDIR = Path('/kaggle/working/Telegram_Movie1_All_Links_No_Channels_v2')
WORKDIR.mkdir(parents=True, exist_ok=True)
# Highly compressed original upload data: all 3,780 outbound records, all 3,751
# original photo references, all non-channel links, metadata and original sender.
# The redundant pre/post caption audit is omitted to fit Kaggle's 1-MB source limit.
ARCHIVE_B64 = '...'
EXPECTED_ARCHIVE_SHA256 = 'c08e14f68db5f865573f54ac9c51c0ce8cbcd3227298b4fd8e61f66e5ec20de4'
EXPECTED_MEMBER_SHA256 = {'result.all_downloads.json': '14a4c048804d826e44a74f7290350b10cc61c78de7fd97e5078d6382558e7e10', 'all_photos.json': '02c788a6bba515b7125c8d7df4b00a42bf6ad0b935a090f7f1cc422abbf51780', 'skipped_records.json': 'b80685f528ebf77f11dcd2ee3f0041ff55afcef778725b1b1896bd6890428257', 'summary.json': 'ec8d06018e5084db7d448e656ee5ab10d2fb764587b455aed0aa7eafa7dfddf0', 'clean_captions.py': '2bca7852a7c09d6a9b7138d6b374da94c9e2a958fd9c68a39dd90928d71da397', 'uploader.py': '26a767d17acc0c5f4e6e409b0b687992512148dd7325787bdac630b32257de83'}
raw_archive = base64.b64decode(ARCHIVE_B64)
if hashlib.sha256(raw_archive).hexdigest() != EXPECTED_ARCHIVE_SHA256:
    raise RuntimeError('Bundled migration-data checksum mismatch: no posts sent.')
with tarfile.open(fileobj=io.BytesIO(lzma.decompress(raw_archive)), mode='r:') as archive:
    found = set()
    for member in archive.getmembers():
        if (not member.isfile() or member.name not in EXPECTED_MEMBER_SHA256
                or member.name in found or Path(member.name).name != member.name):
            raise RuntimeError('Unexpected file in migration bundle: ' + member.name)
        found.add(member.name)
        contents = archive.extractfile(member).read()
        if hashlib.sha256(contents).hexdigest() != EXPECTED_MEMBER_SHA256[member.name]:
            raise RuntimeError('Changed prepared file: ' + member.name)
        destination = WORKDIR / member.name
        if destination.exists() and destination.read_bytes() != contents:
            raise RuntimeError('Prepared file changed; cannot overwrite: ' + member.name)
        if not destination.exists():
            destination.write_bytes(contents)
    if found != set(EXPECTED_MEMBER_SHA256):
        raise RuntimeError('Migration bundle is incomplete.')

def load_module(name, filename):
    spec=importlib.util.spec_from_file_location(name,WORKDIR / filename)
    module=importlib.util.module_from_spec(spec)
    sys.modules[name]=module
    spec.loader.exec_module(module)
    return module

uploader=load_module('movie1_no_channels_uploader','uploader.py')
cleaner=load_module('movie1_channel_cleaner','clean_captions.py')
MESSAGES=json.loads((WORKDIR/'result.all_downloads.json').read_text(encoding='utf-8'))['messages']
MANIFEST=json.loads((WORKDIR/'all_photos.json').read_text(encoding='utf-8'))
SUMMARY=json.loads((WORKDIR/'summary.json').read_text(encoding='utf-8'))
uploader.validate_plan(MESSAGES,MANIFEST)
if (len(MESSAGES)!=3780 or len(MESSAGES)!=SUMMARY['upload_posts']
        or SUMMARY['photo_posts']!=3751 or any(cleaner.remaining_channel_references(m) for m in MESSAGES)):
    raise RuntimeError('Migration data validation failed. NO POSTS SENT.')
print('READY:',len(MESSAGES),'outbound posts; photos:',SUMMARY['photo_posts'],
      '; text:',SUMMARY['text_posts'],'; outgoing download links preserved:',
      SUMMARY['non_channel_link_occurrences_preserved'])
print('NO Telegram messages sent in preparation.')


In [ ]:
#@title 2. Preview original outgoing records (NO Telegram sends)
for item in MESSAGES[:5]:
    txt=uploader.flattened(item)
    print('source:',item['id'],'| photo:',bool(item.get('photo')),'|',
          (txt[:170].replace('\n',' | ') if txt else '(no caption)'))
print('All',SUMMARY['photo_posts'],'original photo records and all original non-channel links are in the upload plan.')
print('Channel promotion links excluded:',SUMMARY['channel_link_occurrences_removed'])


In [ ]:
#@title 3. Verify saved destination ID and automatically post all remaining records
import json, shutil
from pathlib import Path
from urllib.parse import urlparse

if RUN_MODE == 'PREVIEW':
    print('PREVIEW mode: nothing will be posted.')
else:
    from kaggle_secrets import UserSecretsClient
    try:
        bot_token = UserSecretsClient().get_secret('TELEGRAM_BOT_TOKEN')
    except Exception as exc:
        raise RuntimeError('Grant access to existing TELEGRAM_BOT_TOKEN in Kaggle Secrets. No posts sent.') from exc
    if not bot_token:
        raise RuntimeError('TELEGRAM_BOT_TOKEN is empty. No posts sent.')
    API=uploader.TelegramAPI(bot_token)
    del bot_token
    BOT=API.call('getMe')
    if not BOT.get('is_bot'):
        raise RuntimeError('Secret did not identify a Telegram bot. No posts sent.')
    if API.call('getWebhookInfo').get('url'):
        print('Warning: this bot uses a webhook, so recent channel-post updates cannot be read.')
        updates=[]
    else:
        try:
            updates=API.call('getUpdates',{'timeout':0,'limit':100})
        except uploader.ApiRejected as exc:
            print('Bot update queue unavailable; checking saved destination instead:',str(exc))
            updates=[]

    def invite_code(link):
        if not isinstance(link,str): return None
        u=urlparse(link.strip())
        if u.netloc.lower() not in ('t.me','telegram.me','www.t.me','www.telegram.me'):
            return None
        path=u.path.rstrip('/')
        if path.startswith('/+'): return path[2:]
        if path.startswith('/joinchat/'): return path[len('/joinchat/'):]
        return None

    expected_invite=invite_code(TARGET_INVITE_URL)
    if not expected_invite or VERIFICATION_POST != 'MIGRATION_TARGET_' + expected_invite:
        raise RuntimeError('Configured invite and marker do not agree. No posts sent.')

    proof_ids=set()
    update_ids=set()
    for update in updates:
        for key in ('channel_post','edited_channel_post'):
            post=update.get(key) or {}
            chat=post.get('chat') or {}
            if chat.get('type')!='channel' or not isinstance(chat.get('id'),int):
                continue
            cid=chat['id']
            update_ids.add(cid)
            text=(str(post.get('text',''))+'\n'+str(post.get('caption','')))
            if VERIFICATION_POST in (line.strip() for line in text.splitlines()):
                proof_ids.add(cid)
        member_chat=(update.get('my_chat_member') or {}).get('chat') or {}
        if member_chat.get('type')=='channel' and isinstance(member_chat.get('id'),int):
            update_ids.add(member_chat['id'])

    # Only the exact ID from the previous user's migration notebook is eligible.
    # Never automatically switch to another channel based on update order.
    cid=PREVIOUS_NOTEBOOK_CHAT_ID
    if cid == -1001881963160:
        raise RuntimeError('Refusing to post into the original/source channel.')
    try:
        BOT,CHANNEL=uploader.validate_target(API,cid)
    except (uploader.ApiRejected,ValueError) as exc:
        raise RuntimeError('Saved channel is inaccessible or bot cannot post as admin. No posts sent.') from exc
    if CHANNEL.get('id')!=cid or CHANNEL.get('type')!='channel':
        raise RuntimeError('Telegram returned an unexpected chat for the saved ID. No posts sent.')
    actual_title=str(CHANNEL.get('title','')).strip()
    if actual_title.casefold() != EXPECTED_TARGET_TITLE.casefold():
        raise RuntimeError('Channel TITLE changed or does not match saved destination: '
                           + repr(actual_title) + '. No posts sent.')
    if any(proved_id != cid for proved_id in proof_ids):
        raise RuntimeError('Verification marker was received from a DIFFERENT channel; no posts sent.')
    if invite_code(CHANNEL.get('invite_link')) == expected_invite:
        verification='Telegram primary invite matches supplied link'
    elif cid in proof_ids:
        verification='exact verification message received by bot'
    elif ALLOW_PREVIOUS_NOTEBOOK_DESTINATION:
        verification='saved notebook channel ID and title, plus live bot admin permissions'
        print('NOTE: Telegram did not reveal the posted marker/this invite link to the bot.')
        print('Using preconfigured destination. The invite URL itself is NOT independently verified.')
    else:
        raise RuntimeError('Invite/post proof unavailable; refusing to use the saved ID. No posts sent.')

    print('TARGET:',actual_title,'| channel ID:',cid,'| authorization:',verification)
    STATE_PATH=WORKDIR/('progress_'+str(cid)+'.json')
    OUTPUT_CHECKPOINT=Path('/kaggle/working')/STATE_PATH.name
    CACHE=Path('/kaggle/temp/Telegram_Movie1_Photo_Cache')
    if not STATE_PATH.exists():
        input_root=Path('/kaggle/input')
        choices=(sorted(p for p in input_root.rglob(STATE_PATH.name) if p.is_file())
                 if input_root.exists() else [])
        if len(choices)>1:
            raise RuntimeError('Multiple old progress files found; remove duplicate Kaggle input datasets.')
        if len(choices)==1:
            previous=json.loads(choices[0].read_text(encoding='utf-8'))
            if (previous.get('plan_sha256') != uploader.plan_hash(MESSAGES)
                    or previous.get('bot_id') != BOT['id'] or previous.get('chat_id') != cid):
                raise RuntimeError('Attached progress file is for another bot, channel or upload plan.')
            shutil.copyfile(choices[0],STATE_PATH)
            print('Restored prior progress file:',choices[0])
        else:
            print('WARNING: No earlier progress checkpoint was attached.')
            print('Only start fresh if none of these posts have been migrated before!')

    state=uploader.load_state(STATE_PATH,MESSAGES,BOT['id'],cid)
    if state.get('pending'):
        raise RuntimeError('One prior send is uncertain; inspect that post before resuming: '+repr(state['pending']))
    print('Already sent:',len(state['sent']),'/',len(MESSAGES))
    print('SEND START:',RUN_MODE,'target',cid)
    original_atomic_json=uploader.atomic_json
    def save_with_output_copy(path,data):
        original_atomic_json(path,data)
        if Path(path).resolve()==STATE_PATH.resolve():
            shutil.copyfile(STATE_PATH,OUTPUT_CHECKPOINT)
    uploader.atomic_json=save_with_output_copy
    try:
        state=uploader.run_upload(API,MESSAGES,MANIFEST,STATE_PATH,CACHE,
                    BOT['id'],cid,limit=(5 if RUN_MODE=='SEND_5' else 0),interval=3.0)
        print('COMPLETE:',len(state['sent']),'/',len(MESSAGES))
    finally:
        if STATE_PATH.exists():
            shutil.copyfile(STATE_PATH,OUTPUT_CHECKPOINT)
            print('Checkpoint in this Kaggle runtime:',OUTPUT_CHECKPOINT)


## After starting

In Kaggle's **Versions** page inspect the completed version and output file `progress_-1004492918137.json`. `COMPLETE: 3780 / 3780` means every intended post has a confirmed send response. Total run time includes at least ~3h 9m of configured pauses plus downloads/uploads and Telegram waits. Kaggle runtime quotas or interruptions are possible. Do not start a second simultaneous uploader.

**If another error appears**, capture the first printed traceback (no bot token). The previous `No destination channel verified` error should be bypassed by the saved-ID + title + admin fallback only when those checks succeed. The invite-link identity remains unverified when that fallback is used.
